In [2]:
# ============================================
# IRF Analysis (Single Cell - Save Everything)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR

# -----------------------------
# Config
# -----------------------------
DATA_PATH = "./merged_var_input.csv"

VARS = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_DXY",
    "d_UST10Y",
    "dlog_VIX"
]

DATE_COL_CANDIDATES = ["Date", "date", "DATE"]

LAG_ORDER = 1
IRF_HORIZON = 20
BOOTSTRAP_REPL = 1000

# -----------------------------
# Load Data
# -----------------------------
df = pd.read_csv(DATA_PATH)

date_col = None
for c in DATE_COL_CANDIDATES:
    if c in df.columns:
        date_col = c
        break

if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    df = df.set_index(date_col)

df = df[VARS].dropna()

print("Data shape:", df.shape)

# -----------------------------
# Fit VAR
# -----------------------------
model = VAR(df)
results = model.fit(LAG_ORDER)

print(results.summary())

# -----------------------------
# IRF calculation
# -----------------------------
irf = results.irf(IRF_HORIZON)

# -----------------------------
# Save IRF numeric values
# -----------------------------
irf_values = irf.irfs
names = results.names

records = []

for h in range(irf_values.shape[0]):
    for i, resp in enumerate(names):
        for j, imp in enumerate(names):
            records.append({
                "horizon": h,
                "response": resp,
                "impulse": imp,
                "irf": irf_values[h, i, j]
            })

irf_df = pd.DataFrame(records)
irf_df.to_csv("./irf_values.csv", index=False)

print("Saved -> ./irf_values.csv")

# -----------------------------
# IRF Confidence Intervals
# -----------------------------
lower, upper = irf.errband_mc(repl=BOOTSTRAP_REPL)

records = []

for h in range(lower.shape[0]):
    for i, resp in enumerate(names):
        for j, imp in enumerate(names):
            records.append({
                "horizon": h,
                "response": resp,
                "impulse": imp,
                "lower": lower[h, i, j],
                "upper": upper[h, i, j]
            })

ci_df = pd.DataFrame(records)
ci_df.to_csv("./irf_confidence_intervals.csv", index=False)

print("Saved -> ./irf_confidence_intervals.csv")

# -----------------------------
# Plot: full IRF matrix
# -----------------------------
fig = irf.plot(orth=True)
plt.tight_layout()
plt.savefig("./irf_full_matrix.png", dpi=300)
plt.close()

print("Saved -> ./irf_full_matrix.png")

# -----------------------------
# Plot: COPPER -> SOLVPN
# -----------------------------
fig = irf.plot(
    impulse="dlog_COPPER",
    response="dlog_SOLVPN",
    orth=True
)

plt.title("IRF: COPPER shock -> SOLVPN response")
plt.tight_layout()
plt.savefig("./irf_COPPER_to_SOLVPN.png", dpi=300)
plt.close()

print("Saved -> ./irf_COPPER_to_SOLVPN.png")

# -----------------------------
# Plot: SOLVPN -> COPPER
# -----------------------------
fig = irf.plot(
    impulse="dlog_SOLVPN",
    response="dlog_COPPER",
    orth=True
)

plt.title("IRF: SOLVPN shock -> COPPER response")
plt.tight_layout()
plt.savefig("./irf_SOLVPN_to_COPPER.png", dpi=300)
plt.close()

print("Saved -> ./irf_SOLVPN_to_COPPER.png")

print("\nAll IRF results saved in ./ directory")

Data shape: (1325, 5)
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Thu, 05, Mar, 2026
Time:                     16:18:35
--------------------------------------------------------------------
No. of Equations:         5.00000    BIC:                   -39.1280
Nobs:                     1324.00    HQIC:                  -39.2015
Log likelihood:           16617.2    FPE:                9.03375e-18
AIC:                     -39.2456    Det(Omega_mle):     8.83181e-18
--------------------------------------------------------------------
Results for equation dlog_SOLVPN
                    coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------------
const                  0.000383         0.000339            1.128           0.259
L1.dlog_SOLVPN         0.061663         0.034056            1.811           0.070
L1.dlog_COPPER         0.002

c:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Saved -> ./irf_confidence_intervals.csv
Saved -> ./irf_full_matrix.png
Saved -> ./irf_COPPER_to_SOLVPN.png
Saved -> ./irf_SOLVPN_to_COPPER.png

All IRF results saved in ./ directory
